In [ ]:
%pip install -q --upgrade llama-index llama-parse llama-index-llms-gemini llama-index-embeddings-gemini llama-index-vector-stores-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.4/720.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 7.8 MB/s eta 0:00:00


In [ ]:
from pinecone import Pinecone, ServerlessSpec
from llama_index.core import Settings
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.llms.gemini import Gemini

pc = Pinecone(api_key = usedata.get("PINECONE_API_KEY"))

# Setting global parameter
Settings.embed_model = GeminiEmbedding(model_name="models/text-embedding-004",api_key=userdata.get("GOOGLE_API_KEY_1")) # set the embedding model, add models/ before the model name
Settings.llm = Gemini(model_name="models/gemini-2.5-flash-preview-04-17",temperature=1,api_key=userdata.get("GOOGLE_API_KEY_1"))

<ipython-input-7-239256d3dc7d>:11: DeprecationWarning: Call to deprecated class GeminiEmbedding. (Should use `llama-index-embeddings-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/embeddings/google_genai/)
  Settings.embed_model = GeminiEmbedding(model_name="models/text-embedding-004",api_key=userdata.get("GOOGLE_API_KEY_1")) # set the embedding model, add models/ before the model name
<ipython-input-7-239256d3dc7d>:12: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/)
  Settings.llm = Gemini(model_name="models/gemini-2.5-flash-preview-04-17",temperature=1,api_key=userdata.get("GOOGLE_API_KEY_1"))


In [ ]:
from llama_parse import LlamaParse
from google.colab import userdata
from llama_index.core import SimpleDirectoryReader
import nest_asyncio

nest_asyncio.apply()

parser = LlamaParse(
   api_key=userdata.get('LLAMA_CLOUD_API_KEY'),
   system_prompt = """Parse the dataset by converting 'Order Time' and 'Delivery Time' to datetime objects; extract features like hour, day of week, and month from them; encode categorical fields such as restaurant name, location, pizza size/type, and payment category; normalize or scale numerical columns like delivery duration, distance, topping density, and traffic impact; create new features such as 'is weekend', 'is rush hour', and 'delay ratio'; and define the prediction target using 'Delay (min)' for regression or 'Is Delayed' for classification tasks""",
   result_type="markdown"
)

In [ ]:
#convert the xlsx file into csv
import pandas as pd
df = pd.read_excel('/content/pizza_data.xlsx')
df.to_csv('pizza_data.csv', index=False)

In [ ]:
file_extractor = {".csv": parser}
documents = SimpleDirectoryReader(input_files=['/content/pizza_data.csv'], file_extractor=file_extractor).load_data()

Started parsing the file under job_id 88f149a2-34c1-4453-8d81-13e0b6478e89


In [ ]:
documents[0].text

"|Order ID|Restaurant Name|Location|Order Time|Delivery Time|Delivery Duration (min)|Pizza Size|Pizza Type|Toppings Count|Distance (km)|Traffic Level|Payment Method|Is Peak Hour|Is Weekend|Delivery Efficiency (min/km)|Topping Density|Order Month|Payment Category|Estimated Duration (min)|Delay (min)|Is Delayed|Pizza Complexity|Traffic Impact|Order Hour|Restaurant Avg Time|\n|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|\n|ORD001|Domino's|New York, NY|2024-01-05 18:30:00|2024-01-05 18:45:00|15|Medium|Veg|3|2.5|Medium|Card|True|False|6.0|1.2|January|Online|6.0|9.0|False|6|2|18|30.25943396226415|\n|ORD002|Papa John's|Los Angeles, CA|2024-02-14 20:00:00|2024-02-14 20:25:00|25|Large|Non-Veg|4|5.0|High|Wallet|True|False|5.0|0.8|February|Online|12.0|13.0|False|12|3|20|28.18627450980392|\n|ORD003|Little Caesars|Chicago, IL|2024-03-21 12:15:00|2024-03-21 12:35:00|20|Small|Vegan|2|3.0|Low|UPI|False|False|6.666666666666667|0.6666666666666666|M

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

# Initialize SentenceSplitter to split on newline character
node_parser = SentenceSplitter(separator='\n')

# Assuming 'documents' is your list of documents loaded previously
nodes = node_parser.get_nodes_from_documents(documents)

# Now, the 'nodes' list will contain nodes that are split after each newline

In [ ]:
len(nodes)

125

In [ ]:
embedding_dimensions = len(Settings.embed_model.get_text_embedding("Hi"))

In [ ]:
# Create Pinecone vector store
import os
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata

pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))

index_name = "excel-rag"

# Check if index already exists
if index_name not in [index['name'] for index in pc.list_indexes()]:
    # Create index if it doesn't exist
    pc.create_index(
        name=index_name,
        dimension=embedding_dimensions,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Index '{index_name}' created.")
    pinecone_index = pc.Index(index_name)
else:
    pinecone_index = pc.Index(index_name)
    print(f"Index '{index_name}' already exists and has been initialized")

Index 'excel-rag' created.


In [ ]:
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext
from llama_index.core.indices.vector_store.base import VectorStoreIndex


vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
VectorStoreIndex(nodes, storage_context=storage_context,show_progress=True)

Generating embeddings:   0%|          | 0/125 [00:00<?, ?it/s]

Upserted vectors:   0%|          | 0/125 [00:00<?, ?it/s]

In [ ]:
from llama_index.core.indices import load_index_from_storage


pineconeindex = VectorStoreIndex.from_vector_store(vector_store=vector_store)

# create a query engine for the index
query_engine = pineconeindex.as_query_engine()


def get_response(query):
 response = query_engine.query(query)
 return response.response

In [ ]:
get_response("How many orders were delivered by Domino's in total?")

"There were 2 orders delivered by Domino's."

In [ ]:
print(get_response("where are the locations of the deliveries?"))

The locations where deliveries take place include:
*   New York, NY
*   Chicago, IL
*   Los Angeles, CA
*   Houston, TX
*   Phoenix, AZ
*   Miami, FL
*   Omaha, NE
*   Louisville, KY
*   Milwaukee, WI
*   Albuquerque, NM
*   Atlanta, GA


In [ ]:
print(get_response("ORD016 order details"))

The details for order ORD016 are not available.
